# 01 · Data audit

**Question for this notebook:** is the data what we think it is, before anything is estimated on it?

1. What the configured competitions contain
2. Is every match downloaded?
3. Validation: does the score rebuilt from events match the official score?
4. Sanity table: shots, goals, passes and xG per match

Everything here reads what `make ingest` already downloaded, except step 1, which asks StatsBomb's index. Logic that turns out to be useful moves into `src/confidence_shocks/` and gets a test.

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from confidence_shocks.config import ROOT, load_config

# Which run to look at. Switch to leagues_2015_16.yaml for the main analysis.
CONFIG = ROOT / "configs" / "default.yaml"
cfg = load_config(CONFIG)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S")
pd.set_option("display.max_columns", 40)
print(cfg.name, "|", ", ".join(f"{c['name']} {c['season']}" for c in cfg.competitions))

## 1 · What the configured competitions contain

The same table as `make audit`. A full league season has 380 matches (306 for an 18-team league). Far fewer means the free data covers only one club: drop that league from the config.

In [ ]:
from confidence_shocks import pipeline
from confidence_shocks.data import statsbomb as source

audit = pipeline.audit(cfg, source=source)
audit

## 2 · Is every match downloaded?

In [ ]:
matches = pd.read_parquet(cfg.matches_path)
have = {int(p.stem) for p in cfg.events_dir.glob("*.parquet")}
missing = matches[~matches["match_id"].isin(have)]
print(f"{len(matches) - len(missing)} of {len(matches)} matches downloaded")
missing.head()

If anything is missing, run `make ingest` again (with `CONFIG=...` for a non-default config). It resumes where it stopped.

Load every downloaded match into one table. For the league config this is a few million rows, so only the columns used below are kept.

In [ ]:
COLUMNS = ["match_id", "period", "team", "player", "type", "shot_type", "shot_outcome",
           "shot_statsbomb_xg", "pass_outcome", "clock"]

events = pd.concat(
    (pd.read_parquet(cfg.events_dir / f"{m}.parquet", columns=COLUMNS)
     for m in matches["match_id"] if m in have),
    ignore_index=True,
)
events = events.merge(matches[["match_id", "competition", "season"]], on="match_id")
print(f"{len(events):,} events")
events["type"].value_counts().head(15)

## 3 · Validation: rebuild the score from events

The shock table depends on knowing the score at every moment (`score_diff`). If goals were dropped or double-counted, that would be wrong without any error being raised. So count goals per team from the events (shots that went in, plus own goals) and compare with the official result. Penalty shootouts are already removed by `prep`.

In [ ]:
is_goal = ((events["type"] == "Shot") & (events["shot_outcome"] == "Goal")) | (
    events["type"] == "Own Goal For"
)
goals = events[is_goal].groupby(["match_id", "team"]).size()

def rebuilt(row, side):
    return int(goals.get((row["match_id"], row[f"{side}_team"]), 0))

check = matches[["match_id", "competition", "home_team", "away_team", "home_score", "away_score"]].copy()
check = check[check["match_id"].isin(have)]
check["home_events"] = check.apply(rebuilt, side="home", axis=1)
check["away_events"] = check.apply(rebuilt, side="away", axis=1)
check["ok"] = (check["home_events"] == check["home_score"]) & (check["away_events"] == check["away_score"])

print(f"{check['ok'].sum()} of {len(check)} matches agree with the official score")
exceptions = check[~check["ok"]]
exceptions

Every exception needs an explanation before Phase 3. The usual suspects: an own goal recorded under an unexpected event type, extra time handled differently, or a team name that differs between the match list and the events. Write the exceptions and their causes into `data/README.md`.

## 4 · Sanity table

In [ ]:
shots = events[(events["type"] == "Shot") & (events["shot_type"] != "Penalty")]
per_match = pd.DataFrame({
    "matches": events.groupby("competition")["match_id"].nunique(),
    "shots": shots.groupby("competition").size(),
    "goals": shots[shots["shot_outcome"] == "Goal"].groupby("competition").size(),
    "xg": shots.groupby("competition")["shot_statsbomb_xg"].sum(),
    "passes": events[events["type"] == "Pass"].groupby("competition").size(),
})
rates = ["shots", "goals", "xg", "passes"]
per_match[rates] = per_match[rates].div(per_match["matches"], axis=0)
per_match["pass_cmp"] = (
    events[events["type"] == "Pass"].groupby("competition")["pass_outcome"].apply(lambda s: s.isna().mean())
)
per_match.round(2)

**What to expect** for men's top-flight football: roughly 22 to 27 shots, 2.5 to 3 goals (non-penalty a little lower) and 900 to 1,000 passes per match, with completion around 80%. Total xG per match should sit close to non-penalty goals per match. A league far outside these ranges points to a data problem, not a discovery.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(shots["shot_statsbomb_xg"], bins=40)
ax.set_xlabel("xG per shot (non-penalty)")
ax.set_ylabel("shots")
ax.set_title(f"{cfg.name}: most shots are long shots")
plt.show()